In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [45]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import regex as re

# Load Phase 3 combined CSV
ALL_CSV = "/content/drive/MyDrive/MSIS-822-Project/v2/data/03_phase3_features/phase3_all_features_with_split_updated.csv"
df = pd.read_csv(ALL_CSV)

# Identify features
feat_cols = [c for c in df.columns if re.match(r"^f\d{3}_", c)]

# Convert label → AI / Human
df["label_norm"] = df["label"].map({1: "AI", 0: "Human"})


In [46]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 57698 entries, 0 to 57697
Data columns (total 23 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   text                        57698 non-null  object 
 1   text_clean                  57698 non-null  object 
 2   label                       57698 non-null  int64  
 3   model_name                  57698 non-null  object 
 4   split_name                  57698 non-null  object 
 5   f001_total_chars            57698 non-null  int64  
 6   f004_ws_over_C              57698 non-null  float64
 7   f013_hapax_ratio            57698 non-null  float64
 8   f022_entropy_wordfreq       57698 non-null  float64
 9   f025_single_quotes          57698 non-null  int64  
 10  f034_total_sentences        57698 non-null  int64  
 11  f043_num_nouns              57698 non-null  int64  
 12  f046_num_adverbs            57698 non-null  int64  
 13  f055_noun_to_verb_ratio     576

### 1. By Class

In [47]:
# Replace infinities to avoid NaNs in means
safe_df = df.replace([np.inf, -np.inf], np.nan)

# Aggregate: MultiIndex columns (feature, stat)
g = safe_df.groupby("label_norm")[feat_cols].agg(["min", "mean", "max"])

# Build summary with columns like AI_mean, Human_mean, AI_min, ...
stats = ["min", "mean", "max"]
labels = list(g.index)

frames = []
for stat in stats:
    tmp = g.xs(stat, axis=1, level=1).T        # rows=features, cols=labels
    tmp = tmp.rename(columns={lab: f"{lab}_{stat}" for lab in tmp.columns})
    frames.append(tmp)

summary = pd.concat(frames, axis=1)

# Pretty display
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")
display(summary)

label_norm,AI_min,Human_min,AI_mean,Human_mean,AI_max,Human_max
f001_total_chars,170.000,411.000,654.627,740.422,"18,459.000","1,891.000"
f004_ws_over_C,0.101,0.097,0.154,0.160,0.192,0.220
f013_hapax_ratio,0.000,0.000,0.637,0.679,1.000,1.000
f022_entropy_wordfreq,0.310,0.000,5.807,6.119,7.562,7.465
f025_single_quotes,0.000,0.000,0.003,0.013,6.000,8.000
f034_total_sentences,1.000,1.000,4.834,2.698,160.000,16.000
f043_num_nouns,1.000,0.000,47.513,53.328,"1,166.000",152.000
f046_num_adverbs,0.000,0.000,0.264,0.495,13.000,6.000
f055_noun_to_verb_ratio,1.000,0.000,6.181,5.761,93.000,30.000
f064_num_nominatives,2.000,0.000,29.445,36.881,956.000,119.000


### 2. By Split & Class

In [48]:
# --- aggregate: index=(split, class), columns=(feature, stat) ---
g = safe_df.groupby(["split_name", "label_norm"])[feat_cols].agg(["min", "mean", "max"])

# --- build wide matrix: rows=features, cols="<split>_<class>_<stat>" ---
stats  = ["min", "mean", "max"]
splits = list(g.index.get_level_values(0).unique())
classes = list(g.index.get_level_values(1).unique())

frames = []
for stat in stats:
    # extract this stat regardless of MultiIndex level order
    block = g.xs(stat, axis=1, level=1) if stat in g.columns.get_level_values(1) else g.xs(stat, axis=1, level=0)
    block = block.T  # rows=features, cols=(split, class)
    # rename multi-columns to "<split>_<class>_<stat>"
    block.columns = [f"{s}_{c}_{stat}" for (s, c) in block.columns]
    frames.append(block)

summary_split_class = pd.concat(frames, axis=1)

# --- pretty display ---
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")
display(summary_split_class)

,by_polishing_AI_min,by_polishing_Human_min,from_title_AI_min,from_title_Human_min,from_title_and_content_AI_min,from_title_and_content_Human_min,by_polishing_AI_mean,by_polishing_Human_mean,from_title_AI_mean,from_title_Human_mean,from_title_and_content_AI_mean,from_title_and_content_Human_mean,by_polishing_AI_max,by_polishing_Human_max,from_title_AI_max,from_title_Human_max,from_title_and_content_AI_max,from_title_and_content_Human_max
f001_total_chars,170.000,411.000,181.000,411.000,180.000,416.000,706.183,740.693,590.574,740.214,671.245,740.359,"18,459.000","1,844.000","1,603.000","1,891.000","13,235.000","1,891.000"
f004_ws_over_C,0.112,0.097,0.101,0.104,0.124,0.097,0.155,0.160,0.152,0.160,0.155,0.160,0.190,0.220,0.184,0.220,0.192,0.220
f013_hapax_ratio,0.000,0.000,0.000,0.000,0.000,0.000,0.661,0.679,0.647,0.679,0.598,0.680,1.000,1.000,1.000,1.000,1.000,1.000
f022_entropy_wordfreq,4.297,0.000,0.918,0.000,0.310,0.000,5.923,6.117,5.722,6.120,5.776,6.121,7.034,7.465,6.723,7.465,7.562,7.465
f025_single_quotes,0.000,0.000,0.000,0.000,0.000,0.000,0.002,0.012,0.006,0.012,0.001,0.013,4.000,8.000,6.000,8.000,2.000,8.000
f034_total_sentences,1.000,1.000,1.000,1.000,1.000,1.000,4.780,2.687,4.367,2.699,5.432,2.708,160.000,16.000,15.000,16.000,101.000,16.000
f043_num_nouns,9.000,0.000,1.000,0.000,6.000,0.000,50.952,53.324,42.815,53.307,49.112,53.357,882.000,147.000,118.000,152.000,"1,166.000",152.000
f046_num_adverbs,0.000,0.000,0.000,0.000,0.000,0.000,0.233,0.492,0.243,0.493,0.323,0.500,6.000,6.000,4.000,6.000,13.000,6.000
f055_noun_to_verb_ratio,1.000,0.000,1.250,0.000,1.250,0.000,6.111,5.765,6.272,5.746,6.153,5.773,60.000,30.000,78.000,30.000,93.000,30.000
f064_num_nominatives,3.000,0.000,2.000,0.000,3.000,0.000,31.463,36.866,26.683,36.886,30.389,36.892,956.000,119.000,74.000,119.000,522.000,119.000


In [58]:
# Aggregate: (split_name, model_name) × features × (min, mean, max)
g = safe_df.groupby(["split_name", "model_name"])[feat_cols].agg(["min", "mean", "max"])

# Extract list of splits and stats
splits = g.index.get_level_values(0).unique().tolist()
stats = ["min", "mean", "max"]

# Loop through each split and display separately
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")
for split in splits:
    print(f"\n=== Split: {split.upper()} ===\n")

    # Subset the current split
    g_split = g.loc[split]

    frames = []
    for stat in stats:
        # Extract stat for this split
        block = g_split.xs(stat, axis=1, level=1) if stat in g_split.columns.get_level_values(1) else g_split.xs(stat, axis=1, level=0)
        block = block.T  # rows = features, cols = models
        block.columns = [f"{model}_{stat}" for model in block.columns]
        frames.append(block)

    summary_split = pd.concat(frames, axis=1)
    summary_split = summary_split.round(3)
    display(summary_split)


=== Split: BY_POLISHING ===



,allam_min,human_min,jais_min,llama_min,openai_min,allam_mean,human_mean,jais_mean,llama_mean,openai_mean,allam_max,human_max,jais_max,llama_max,openai_max
f001_total_chars,250.000,411.000,170.000,286.000,556.000,672.624,740.693,435.236,649.703,"1,067.638","18,459.000","1,844.000","1,242.000","1,393.000","1,427.000"
f004_ws_over_C,0.134,0.097,0.123,0.112,0.137,0.154,0.160,0.155,0.156,0.154,0.185,0.220,0.190,0.189,0.172
f013_hapax_ratio,0.001,0.000,0.338,0.000,0.452,0.632,0.679,0.722,0.633,0.656,0.962,1.000,1.000,0.922,0.854
f022_entropy_wordfreq,4.827,0.000,4.297,4.906,5.649,5.845,6.117,5.415,5.854,6.578,6.784,7.465,6.793,6.949,7.034
f025_single_quotes,0.000,0.000,0.000,0.000,0.000,0.001,0.012,0.006,0.001,0.001,2.000,8.000,4.000,2.000,2.000
f034_total_sentences,2.000,1.000,1.000,1.000,4.000,5.279,2.687,3.407,4.058,6.383,160.000,16.000,10.000,14.000,11.000
f043_num_nouns,14.000,0.000,9.000,14.000,43.000,49.391,53.324,30.541,47.297,76.610,882.000,147.000,100.000,119.000,129.000
f046_num_adverbs,0.000,0.000,0.000,0.000,0.000,0.118,0.492,0.127,0.249,0.439,6.000,6.000,4.000,4.000,4.000
f055_noun_to_verb_ratio,1.909,0.000,1.000,1.929,2.115,6.042,5.765,5.697,7.139,5.561,32.000,30.000,28.000,60.000,15.200
f064_num_nominatives,8.000,0.000,3.000,10.000,17.000,30.168,36.866,20.443,29.119,46.142,956.000,119.000,65.000,76.000,76.000



=== Split: FROM_TITLE ===



,allam_min,human_min,jais_min,llama_min,openai_min,allam_mean,human_mean,jais_mean,llama_mean,openai_mean,allam_max,human_max,jais_max,llama_max,openai_max
f001_total_chars,252.000,411.000,181.000,278.000,652.000,504.218,740.214,404.237,639.516,814.251,"1,349.000","1,891.000",845.000,"1,603.000",965.000
f004_ws_over_C,0.130,0.104,0.126,0.101,0.132,0.152,0.160,0.152,0.155,0.150,0.178,0.220,0.184,0.179,0.168
f013_hapax_ratio,0.259,0.000,0.133,0.000,0.474,0.649,0.679,0.729,0.531,0.679,0.955,1.000,1.000,0.870,0.864
f022_entropy_wordfreq,4.746,0.000,3.572,0.918,5.194,5.568,6.120,5.354,5.677,6.289,6.453,7.465,6.259,6.723,6.676
f025_single_quotes,0.000,0.000,0.000,0.000,0.000,0.001,0.012,0.009,0.001,0.012,2.000,8.000,6.000,2.000,6.000
f034_total_sentences,2.000,1.000,1.000,1.000,3.000,4.173,2.699,3.477,4.589,5.228,14.000,16.000,9.000,15.000,8.000
f043_num_nouns,17.000,0.000,10.000,1.000,22.000,37.814,53.307,27.830,46.483,59.127,104.000,152.000,60.000,118.000,84.000
f046_num_adverbs,0.000,0.000,0.000,0.000,0.000,0.171,0.493,0.196,0.350,0.253,4.000,6.000,4.000,3.000,3.000
f055_noun_to_verb_ratio,2.250,0.000,1.250,1.923,2.812,6.417,5.746,5.678,6.761,6.229,14.000,30.000,36.000,78.000,33.500
f064_num_nominatives,9.000,0.000,4.000,2.000,18.000,23.407,36.886,18.258,29.951,35.110,65.000,119.000,42.000,74.000,55.000



=== Split: FROM_TITLE_AND_CONTENT ===



,allam_min,human_min,jais_min,llama_min,openai_min,allam_mean,human_mean,jais_mean,llama_mean,openai_mean,allam_max,human_max,jais_max,llama_max,openai_max
f001_total_chars,187.000,416.000,180.000,257.000,567.000,622.633,740.359,666.724,652.724,742.988,"13,235.000","1,891.000","2,968.000","1,636.000",920.000
f004_ws_over_C,0.128,0.097,0.130,0.124,0.134,0.153,0.160,0.158,0.157,0.152,0.192,0.220,0.189,0.183,0.177
f013_hapax_ratio,0.000,0.000,0.000,0.000,0.462,0.603,0.680,0.595,0.525,0.672,1.000,1.000,1.000,0.925,0.853
f022_entropy_wordfreq,4.419,0.000,3.367,0.310,5.478,5.637,6.121,5.733,5.610,6.125,7.004,7.465,7.562,7.117,6.617
f025_single_quotes,0.000,0.000,0.000,0.000,0.000,0.000,0.013,0.003,0.000,0.002,0.000,8.000,2.000,0.000,2.000
f034_total_sentences,2.000,1.000,1.000,1.000,3.000,5.556,2.708,5.748,5.085,5.341,71.000,16.000,34.000,101.000,9.000
f043_num_nouns,12.000,0.000,11.000,6.000,34.000,46.761,53.357,45.596,48.880,55.209,"1,166.000",152.000,194.000,137.000,80.000
f046_num_adverbs,0.000,0.000,0.000,0.000,0.000,0.162,0.500,0.493,0.322,0.318,5.000,6.000,13.000,11.000,4.000
f055_noun_to_verb_ratio,2.000,0.000,1.250,1.625,2.333,6.234,5.773,4.759,7.692,5.930,30.500,30.000,15.667,93.000,18.000
f064_num_nominatives,7.000,0.000,8.000,3.000,14.000,26.790,36.892,33.224,29.396,32.154,522.000,119.000,170.000,111.000,53.000
